# Bloque 1 — benchmark de un día de pronósticos AIFS Single v1.1 (ERA5)

Mide, para un día de 4 ciclos (00/06/12/18 UTC, 0–72 h cada 6 h):

- tiempo de pared por fase;
- RAM: RSS pico por fase;
- VRAM: pico de PyTorch y uso del dispositivo según `nvidia-smi`;
- volumen de las salidas NetCDF.

Con eso extrapola a la campaña completa: mayo–noviembre 2000–2025, 22 256 pronósticos en 182 tareas año-mes.

**Ejecución**

1. **ERA5 del día.** Se necesita una CPU con internet y `~/.cdsapirc`. Si no existe el GRIB mensual del mes de prueba, la celda §1 descarga solo los tiempos del día. Esto puede correrse en un nodo de login (celdas 1–5) y después reiniciar el kernel en el nodo GPU.
2. **Jupyter en el nodo GPU.** Lanzarlo en la misma partición que se pedirá en producción, con margen de RAM para no truncar la medición:
   `sbatch --cpus-per-task=8 --mem=64G workflows/slurm/run_aifs_jupyter.slurm`.
   Las opciones de línea de comandos sustituyen las directivas `#SBATCH` del script. Si se cambia `--partition` (gpu vs gpu-h100), la extrapolación vale solo para esa partición.
3. **Cifra final, sin interfaz.** Ejecutar `jupyter nbconvert --to notebook --execute` dentro de `apptainer exec --nv` en un sbatch y contrastar con `sacct -j <id> --format=Elapsed,MaxRSS`.

**Notas**

- **Costo fijo.** El primer ciclo incluye la carga perezosa del modelo y el calentamiento de CUDA. Se trata como costo fijo por tarea; el costo estacionario es la mediana de los ciclos 1–3. Tres ciclos son una muestra pequeña: si el coeficiente de variación supera ~10 %, repetir con otro día.
- **Lectura de ERA5.** Si el GRIB es de un solo día y no del mes completo, el tiempo de lectura es una cota inferior. `earthkit` indexa archivos mensuales de ~10⁴ mensajes.
- **t0.** Se toma del estado de entrada (ERA5 → N320 → 0.25°), ya que `runner.run` puede no devolverlo. tp, cp y tcc quedan como NaN en t0.
- **Dos esquemas de salida.** Se escriben ambos para decidir entre ellos:
  - `split`: archivo regional para SwAIther más archivo extendido para TempestExtremes, como en el YAML;
  - `single`: todas las variables en el dominio extendido.

In [ ]:
import os

# Deben fijarse antes de inicializar CUDA
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("ANEMOI_INFERENCE_NUM_CHUNKS", "16")  # reduce VRAM (run_AIFS_v1.1.ipynb, ECMWF)

import datetime as dt
import json
import math
import platform
import subprocess
import sys
import threading
import time
from contextlib import contextmanager
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import yaml

# Usar el src del repo, no la copia instalada al construir el contenedor
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO / "src"))

import torch
import earthkit.regrid as ekr
from anemoi.inference.runners.simple import SimpleRunner

import aitchazard
from aitchazard.block1.dates import analysis_times, group_by_year_month, init_dates
from aitchazard.block1.era5_inputs import ERA5Local, checkpoint_input_variables, compare_inputs, era5_path, field_to_n320
from aitchazard.block1.regrid import Regridder
from aitchazard.block1.era5_params import GROUP_PARAMS, build_monthly_cds_request, translate_params_to_cds

print("aitchazard:", aitchazard.__file__)

In [ ]:
CFG_PATH = REPO / "conf/aitchazard_mexico/block1_aifs_single_v1_1.yaml"
CFG = yaml.safe_load(CFG_PATH.read_text())
SCHED = CFG["schedule"]
LAG_H = CFG["initial_conditions"]["lag_hours"]

BENCH_DAY = dt.date.fromisoformat(os.environ.get("BENCH_DAY", "2005-05-15"))  # evitar el día 1: t-6 h del ciclo 00 cae en el mes previo
INITS = [dt.datetime(BENCH_DAY.year, BENCH_DAY.month, BENCH_DAY.day, h) for h in (0, 6, 12, 18)]
LEAD_H = SCHED["lead_time_hours"]
STEP_H = SCHED["output_step_hours"]
LEADS = np.arange(0, LEAD_H + 1, STEP_H)  # t0 + 12 pasos
N_T = LEADS.size

WORK = Path("/work/FAC/FGSE/IDYST/tbeucler/default/apereze")
CHECKPOINT = WORK / "aifs/checkpoints/aifs-single-mse-1.1.ckpt"  # nombre del .ckpt en ecmwf/aifs-single-1.1; ajustar ruta
JOB_ID = os.environ.get("SLURM_JOB_ID", "local")
OUT = WORK / "benchmarks/block1" / f"{BENCH_DAY:%Y%m%d}_{JOB_ID}"

# ERA5: GRIB mensual de producción si existe; si no, GRIB de un día en el directorio de benchmark
MONTH_REF = dt.datetime(BENCH_DAY.year, BENCH_DAY.month, 1)
ERA5_PROD = Path(CFG["initial_conditions"]["root"])
ERA5_PROD = ERA5_PROD if ERA5_PROD.is_absolute() else REPO / ERA5_PROD
ERA5_BENCH = WORK / "benchmarks/era5_day"
HAS_MONTH = all(era5_path(ERA5_PROD, MONTH_REF, g).exists() for g in GROUP_PARAMS)
ERA5_ROOT = ERA5_PROD if HAS_MONTH else ERA5_BENCH
STAGE_ERA5 = not all(era5_path(ERA5_ROOT, MONTH_REF, g).exists() for g in GROUP_PARAMS)

# Dominios del YAML; descomentar para evaluar un dominio extendido mayor
DOMAINS = {k: {"lat": list(v["lat"]), "lon": list(v["lon"]), "variables": list(v["variables"])}
           for k, v in CFG["output"]["domains"].items()}
# DOMAINS["tempest"]["lon"] = [220.0, 330.0]

UNION_VARS = list(dict.fromkeys(DOMAINS["regional"]["variables"] + DOMAINS["tempest"]["variables"]))
LAYOUTS = {
    "split": {"regional": DOMAINS["regional"], "tempest": DOMAINS["tempest"]},
    "single": {"extended": {**DOMAINS["tempest"], "variables": UNION_VARS}},
}
COMPLEVEL = CFG["output"]["compression"]["complevel"]
COMPLEVEL_TEST = (1, 4, 9)  # solo ciclo 0, esquema single: tamaño frente a tiempo de escritura

OUT.mkdir(parents=True, exist_ok=True)
print(f"ERA5_ROOT = {ERA5_ROOT}  (mes completo: {HAS_MONTH}; descargar: {STAGE_ERA5})")
print("variables (unión):", UNION_VARS)
print("salida:", OUT)

In [ ]:
GIB = 2**30
CUDA = torch.cuda.is_available()


def _proc_status_gib(key: str) -> float:
    # VmRSS (actual) o VmHWM (pico) del proceso, en GiB
    for line in Path("/proc/self/status").read_text().splitlines():
        if line.startswith(key):
            return int(line.split()[1]) * 1024 / GIB
    return float("nan")


def reset_peak_rss() -> None:
    # Reinicia VmHWM al RSS actual (Linux >= 4.0)
    try:
        Path("/proc/self/clear_refs").write_text("5")
    except OSError:
        pass


def gpu_sync() -> None:
    if CUDA:
        torch.cuda.synchronize()


RECORDS: list[dict] = []


@contextmanager
def phase(name: str, **meta):
    # Tiempo de pared, tiempo de CPU (suma de hilos), RSS pico y VRAM pico de un bloque
    gpu_sync()
    if CUDA:
        torch.cuda.reset_peak_memory_stats()
    reset_peak_rss()
    w0, c0 = time.perf_counter(), time.process_time()
    yield
    gpu_sync()
    RECORDS.append({
        "phase": name, **meta,
        "wall_s": time.perf_counter() - w0,
        "cpu_s": time.process_time() - c0,  # cpu_s / wall_s ~ núcleos efectivos
        "rss_gib": _proc_status_gib("VmRSS"),
        "rss_peak_gib": _proc_status_gib("VmHWM"),
        "vram_peak_alloc_gib": torch.cuda.max_memory_allocated() / GIB if CUDA else np.nan,
        "vram_peak_reserved_gib": torch.cuda.max_memory_reserved() / GIB if CUDA else np.nan,
    })


class GPUSampler:
    # Muestrea memoria usada y utilización del dispositivo con nvidia-smi (incluye contexto CUDA y caché)
    def __init__(self, dt_s: float = 0.5):
        self.dt_s, self.rows, self._stop = dt_s, [], threading.Event()
        self.sel = []
        try:
            uuid = str(torch.cuda.get_device_properties(0).uuid)
            self.sel = ["-i", uuid if uuid.startswith("GPU-") else f"GPU-{uuid}"]
        except Exception:
            pass

    def _query(self):
        cmd = ["nvidia-smi", "--query-gpu=memory.used,utilization.gpu", "--format=csv,noheader,nounits", *self.sel]
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=5).stdout.strip().splitlines()
        mem, util = (float(x) for x in out[0].split(","))
        return mem / 1024, util

    def _loop(self):
        t0 = time.perf_counter()
        while not self._stop.is_set():
            try:
                self.rows.append((time.perf_counter() - t0, *self._query()))
            except Exception:
                if self.sel:          # reintentar sin selector de dispositivo
                    self.sel = []
            self._stop.wait(self.dt_s)

    def start(self):
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def stop(self) -> pd.DataFrame:
        self._stop.set()
        self._thread.join()
        return pd.DataFrame(self.rows, columns=["t_s", "mem_used_gib", "util_pct"])

In [ ]:
def _ver(pkg: str):
    try:
        return metadata.version(pkg)
    except metadata.PackageNotFoundError:
        return None


ENV = {
    "python": platform.python_version(),
    "packages": {p: _ver(p) for p in ("torch", "anemoi-inference", "anemoi-models", "anemoi-utils",
                                      "earthkit-data", "earthkit-regrid", "xarray", "netCDF4", "numpy")},
    "gpu": torch.cuda.get_device_properties(0).name if CUDA else None,
    "vram_total_gib": torch.cuda.get_device_properties(0).total_memory / GIB if CUDA else None,
    "cpus_affinity": len(os.sched_getaffinity(0)),
    "torch_threads": torch.get_num_threads(),
    "slurm": {k: os.environ.get(k) for k in ("SLURM_JOB_ID", "SLURM_JOB_PARTITION", "SLURM_CPUS_PER_TASK",
                                             "SLURM_MEM_PER_NODE", "SLURM_JOB_NODELIST")},
    "env": {k: os.environ.get(k) for k in ("ANEMOI_INFERENCE_NUM_CHUNKS", "PYTORCH_CUDA_ALLOC_CONF", "OMP_NUM_THREADS")},
    "era5_root": str(ERA5_ROOT),
    "era5_full_month": HAS_MONTH,
    "bench_day": str(BENCH_DAY),
    "regrid": "batched-box (earthkit-regrid matrices)",
}
print(json.dumps(ENV, indent=2))

## 1. ERA5 del día de prueba

Descarga, solo si falta, los tiempos de análisis del día en el esquema mensual que lee `ERA5Local`: `{root}/YYYY/MM/{sfc,pl,soil}.grib`.

Mide los bytes de GRIB por tiempo de análisis para estimar el almacenamiento de entradas. El tiempo de descarga del CDS depende de su cola, no de Curnagl, y no entra en la solicitud de GPU.

In [ ]:
def stage_day(root: Path) -> None:
    import cdsapi
    times = analysis_times(INITS, LAG_H)
    assert {(t.year, t.month) for t in times} == {(BENCH_DAY.year, BENCH_DAY.month)}, "elegir un día distinto del 1"
    days = sorted({f"{t.day:02d}" for t in times})
    hours = sorted({f"{t.hour:02d}:00" for t in times})  # el CDS hace producto cruzado días x horas
    client = cdsapi.Client()
    for group, (params, levels) in GROUP_PARAMS.items():
        target = era5_path(root, MONTH_REF, group)
        if target.exists():
            continue
        dataset, variables, pl = translate_params_to_cds(source=group, param=params, levelist=levels)
        req = build_monthly_cds_request(year=BENCH_DAY.year, month=BENCH_DAY.month, days=days, hours=hours,
                                        variables=variables, pressure_levels=pl,
                                        grid=tuple(CFG["initial_conditions"]["grid"]))
        target.parent.mkdir(parents=True, exist_ok=True)
        tmp = target.with_suffix(".grib.tmp")
        with phase("cds_download", group=group):
            client.retrieve(dataset, req, str(tmp))
        tmp.rename(target)  # atómico


def n_analysis_times(path: Path) -> int:
    # Tiempos de análisis distintos en un GRIB (contados sobre 2t)
    import earthkit.data as ekd
    meta = ekd.from_source("file", str(path)).sel(shortName="2t").metadata(["dataDate", "dataTime"])
    return len({tuple(m) for m in meta})


if STAGE_ERA5:
    stage_day(ERA5_ROOT)

GRIB_BYTES = {g: era5_path(ERA5_ROOT, MONTH_REF, g).stat().st_size for g in GROUP_PARAMS}
N_TIMES_FILE = n_analysis_times(era5_path(ERA5_ROOT, MONTH_REF, "sfc"))
ERA5_BYTES_PER_TIME = sum(GRIB_BYTES.values()) / N_TIMES_FILE
print({g: f"{b / 1e9:.2f} GB" for g, b in GRIB_BYTES.items()}, f"| {N_TIMES_FILE} tiempos")
print(f"ERA5: {ERA5_BYTES_PER_TIME / 1e6:.0f} MB por tiempo de análisis")

## 2. Modelo

`SimpleRunner` puede cargar el checkpoint de forma perezosa. En ese caso, la carga aparece en el primer paso del ciclo 0 y no aquí; la extrapolación la trata igual, como costo fijo.

In [ ]:
SAMPLER = GPUSampler().start()
CKPT = str(CHECKPOINT) if CHECKPOINT.exists() else {"huggingface": CFG["model"]["huggingface_repo"]}
with phase("load_checkpoint"):
    runner = SimpleRunner(CKPT, device="cuda")
print("checkpoint:", CKPT)

## 3. Malla, recorte y escritura

Interpolación N320 → 0.25° solo en la caja envolvente de todos los dominios. Se usan las mismas matrices de `earthkit-regrid`, pero recortadas a las filas de la caja y aplicadas a todas las variables en un solo producto disperso (`aitchazard.block1.regrid.Regridder`). La lectura de ERA5 hace lo mismo para 0.25° → N320. En el ciclo 0 se verifica que ambas rutas coinciden con `earthkit.regrid.interpolate`. Los dominios son subconjuntos de la caja.

Escritura: float32, zlib, un fragmento por tiempo de pronóstico, escritura atómica. En el NetCDF, 10u/10v se renombran como u10/v10.

In [ ]:
G = 9.80665
RES = 0.25
LAT = 90.0 - RES * np.arange(721)   # N -> S
LON = RES * np.arange(1440)         # 0-360
UNITS = {"tp": "m", "cp": "m", "tcc": "1", "10u": "m s-1", "10v": "m s-1", "msl": "Pa",
         "q_500": "kg kg-1", "q_850": "kg kg-1", "t_500": "K", "t_850": "K",
         "z_300": "m2 s-2", "z_500": "m2 s-2", "z_850": "m2 s-2", "zs": "m", "lsm": "1"}
NC_NAME = {"10u": "u10", "10v": "v10"}


def box(dom: dict) -> tuple[slice, slice]:
    # Índices de un dominio en la malla regular global
    (la0, la1), (lo0, lo1) = sorted(dom["lat"]), sorted(dom["lon"])
    assert 0 <= lo0 < lo1 < 360, "dominio cruza 0°"
    return (slice(int(round((90 - la1) / RES)), int(round((90 - la0) / RES)) + 1),
            slice(int(round(lo0 / RES)), int(round(lo1 / RES)) + 1))


ALL_DOMS = [d for lay in LAYOUTS.values() for d in lay.values()]
BBOX = {"lat": [min(min(d["lat"]) for d in ALL_DOMS), max(max(d["lat"]) for d in ALL_DOMS)],
        "lon": [min(min(d["lon"]) for d in ALL_DOMS), max(max(d["lon"]) for d in ALL_DOMS)]}
BI, BJ = box(BBOX)
BLAT, BLON = LAT[BI], LON[BJ]


def sub_index(dom: dict) -> tuple[slice, slice]:
    # Índices de un dominio relativos a la caja envolvente
    i, j = box(dom)
    return slice(i.start - BI.start, i.stop - BI.start), slice(j.start - BJ.start, j.stop - BJ.start)


def n_points(dom: dict) -> int:
    i, j = sub_index(dom)
    return (i.stop - i.start) * (j.stop - j.start)


def n320_to_bbox(values) -> np.ndarray:
    # Referencia: interpolación global con earthkit y recorte (lento; solo para verificar)
    reg = ekr.interpolate(np.asarray(values), {"grid": "N320"}, {"grid": (RES, RES)})
    return reg[BI, BJ].astype(np.float32)


REGRID = Regridder.from_grids("N320", "LL025", box=(BI, BJ))  # mismos pesos, solo filas de la caja


def n320_many_to_bbox(fields: dict, names) -> dict:
    # Todas las variables en un solo producto disperso, solo en la caja envolvente
    names = [v for v in names if v in fields]
    block = REGRID(np.stack([np.asarray(fields[v]) for v in names]), dtype=np.float32)
    return dict(zip(names, block))


def _atomic_netcdf(ds: xr.Dataset, path: Path, complevel: int, chunks) -> int:
    enc = {n: {"zlib": True, "complevel": complevel, "dtype": "float32", "chunksizes": chunks(ds[n])}
           for n in ds.data_vars}
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    ds.to_netcdf(tmp, encoding=enc, engine="netcdf4")
    tmp.rename(path)
    return path.stat().st_size


def write_forecast(path: Path, data: dict, dom: dict, t0: dt.datetime, complevel: int) -> int:
    # Un pronóstico (lead_time, latitude, longitude); devuelve bytes en disco
    i, j = sub_index(dom)
    t0n = np.datetime64(t0, "ns")
    ds = xr.Dataset(
        {NC_NAME.get(v, v): (("lead_time", "latitude", "longitude"), data[v][:, i, j], {"units": UNITS[v]})
         for v in dom["variables"]},
        coords={"lead_time": ("lead_time", LEADS.astype("int32"), {"units": "hours"}),
                "valid_time": ("lead_time", t0n + LEADS.astype("timedelta64[h]")),
                "init_time": t0n,
                "latitude": ("latitude", BLAT[i], {"units": "degrees_north"}),
                "longitude": ("longitude", BLON[j], {"units": "degrees_east"})},
        attrs={"model": "aifs-single-1.1", "initial_conditions": "ERA5 (CDS)",
               "accumulation": "tp, cp: pendiente de verificar (ver diagnóstico)"},
    )
    return _atomic_netcdf(ds, path, complevel, lambda da: (1, *da.shape[1:]))


def write_static(path: Path, static: dict, dom: dict) -> int:
    i, j = sub_index(dom)
    ds = xr.Dataset({k: (("latitude", "longitude"), v[i, j], {"units": UNITS[k]}) for k, v in static.items()},
                    coords={"latitude": BLAT[i], "longitude": BLON[j]})
    return _atomic_netcdf(ds, path, COMPLEVEL, lambda da: da.shape)


for lay, doms in LAYOUTS.items():
    for name, dom in doms.items():
        print(f"{lay}/{name}: {len(dom['variables'])} variables, {n_points(dom)} puntos")

## 4. Ejecución de los 4 ciclos

Por paso se registra:

- `infer_s`: tiempo hasta que `runner.run` entrega el estado, con sincronización de GPU;
- `post_s`: regrid más recorte de las variables necesarias.

El pronóstico se procesa en flujo, como en producción. Retener los ~100 campos N320 de los 12 estados ocuparía ~2.6 GB por ciclo.

In [ ]:
era5 = ERA5Local(ERA5_ROOT, lag_h=LAG_H)
STEPS, FILES, CLTEST, CYCLES = [], [], [], []
DIAG = {"t0_yielded": False, "missing_outputs": set(), "acc": {"tp": [], "cp": []}, "input_check": None, "regrid_check": None}


def run_forecast(state0: dict, t0: dt.datetime, data: dict, ic: int) -> list[dict]:
    # Itera runner.run en flujo: regrid y recorte de cada estado al recibirlo
    rows, it = [], runner.run(input_state=state0, lead_time=LEAD_H)
    while True:
        gpu_sync()
        w0 = time.perf_counter()
        try:
            st = next(it)
        except StopIteration:
            return rows
        gpu_sync()
        w1 = time.perf_counter()
        lead = int((pd.Timestamp(st["date"]) - pd.Timestamp(t0)).total_seconds() // 3600)
        fields = st["fields"]
        DIAG["missing_outputs"] |= set(UNION_VARS) - set(fields)
        if lead == 0:
            DIAG["t0_yielded"] = True
        else:
            for v, arr in n320_many_to_bbox(fields, UNION_VARS).items():
                data[v][lead // STEP_H] = arr
            if ic == 0:
                for v in DIAG["acc"]:
                    if v in fields:
                        DIAG["acc"][v].append(np.array(fields[v], dtype=np.float64))
        rows.append({"cycle": ic, "lead_h": lead, "infer_s": w1 - w0, "post_s": time.perf_counter() - w1,
                     "vram_reserved_gib": torch.cuda.memory_reserved() / GIB if CUDA else np.nan,
                     "rss_gib": _proc_status_gib("VmRSS")})


def check_regrid(last: dict, reader, t0: dt.datetime, tol: float = 1e-5) -> dict:
    # Compara las rutas por lotes contra earthkit.regrid.interpolate, campo por campo
    out = {}
    for v in ("msl", "2t", "z_500"):
        if v in last:
            ref, new = n320_to_bbox(last[v]), n320_many_to_bbox(last, [v])[v]
            out[f"N320->0.25 {v}"] = float(np.max(np.abs(new - ref)) / (np.max(np.abs(ref)) + 1e-12))
    raw = reader._source(t0, "sfc").sel(dataDate=int(t0.strftime("%Y%m%d")), dataTime=t0.hour * 100, shortName="msl")
    if len(raw):
        ref = field_to_n320(raw[0])
        out["0.25->N320 msl"] = float(np.max(np.abs(last["msl"] - ref)) / (np.max(np.abs(ref)) + 1e-12))
    bad = {k: e for k, e in out.items() if not e <= tol}
    assert not bad, f"la interpolación por lotes difiere de earthkit: {bad}"
    return out


for ic, t0 in enumerate(INITS):
    c0 = time.perf_counter()
    with phase("input_state", cycle=ic):
        state0 = era5.input_state(t0)   # ciclo 0: lee t-6 h y t0; después t-6 h viene de la caché
    f0 = state0["fields"]
    if ic == 0:
        try:
            DIAG["input_check"] = compare_inputs(f0, checkpoint_input_variables(runner))
        except AttributeError as err:
            DIAG["input_check"] = f"omitido: {err}"
        print("input check:", DIAG["input_check"])

    data = {v: np.full((N_T, BLAT.size, BLON.size), np.nan, np.float32) for v in UNION_VARS}
    with phase("post_t0", cycle=ic):
        last = {k: v[-1] for k, v in f0.items()}
        for v, arr in n320_many_to_bbox(last, UNION_VARS).items():
            data[v][0] = arr   # t0 desde ERA5; tp, cp, tcc quedan NaN
        if ic == 0:
            st = n320_many_to_bbox(last, ["z", "lsm"])
            STATIC = {"zs": st["z"] / G, "lsm": st["lsm"]}
    if ic == 0:
        DIAG["regrid_check"] = check_regrid(last, era5, t0)
        print("verificación de la interpolación por lotes:", DIAG["regrid_check"])

    with phase("forecast", cycle=ic):
        steps = run_forecast(state0, t0, data, ic)
    STEPS += steps

    with phase("write", cycle=ic):
        for lay, doms in LAYOUTS.items():
            for name, dom in doms.items():
                w0 = time.perf_counter()
                nbytes = write_forecast(OUT / lay / name / f"aifs11_{name}_{t0:%Y%m%d%H}.nc", data, dom, t0, COMPLEVEL)
                FILES.append({"cycle": ic, "layout": lay, "domain": name, "write_s": time.perf_counter() - w0,
                              "bytes": nbytes, "raw_bytes": 4 * N_T * len(dom["variables"]) * n_points(dom)})

    if ic == 0:
        for lay, doms in LAYOUTS.items():
            for name, dom in doms.items():
                write_static(OUT / lay / name / f"static_{name}.nc", STATIC, dom)
        dom = LAYOUTS["single"]["extended"]
        for cl in COMPLEVEL_TEST:
            w0 = time.perf_counter()
            nbytes = write_forecast(OUT / "complevel_test" / f"cl{cl}.nc", data, dom, t0, cl)
            CLTEST.append({"complevel": cl, "write_s": time.perf_counter() - w0, "bytes": nbytes})

    CYCLES.append({"cycle": ic, "init": t0, "n_states": len(steps), "wall_s": time.perf_counter() - c0})
    print(f"ciclo {ic} ({t0:%Y-%m-%d %H} UTC): {len(steps)} estados, {CYCLES[-1]['wall_s']:.1f} s")
    del data, state0

GPU_TRACE = SAMPLER.stop()

## 5. Diagnósticos

Responden tres verificaciones pendientes:

- si `runner.run` entrega t0;
- si tcc y cp están entre las salidas;
- la convención de acumulación de tp y cp.

Para la acumulación: un campo acumulado desde t0 es no decreciente en casi todos los puntos, y su media global crece de forma aproximadamente lineal con el número de pasos (razón ~12 a 72 h). Un campo por paso de 6 h tiene una razón ~1.

In [ ]:
print("estados por ciclo:", [c["n_states"] for c in CYCLES], f"(esperados {LEAD_H // STEP_H} sin t0)")
print("¿runner.run entrega t0?:", DIAG["t0_yielded"])
print("variables requeridas ausentes en la salida:", sorted(DIAG["missing_outputs"]) or "ninguna")

for v, seq in DIAG["acc"].items():
    if len(seq) < 2:
        print(f"{v}: sin datos suficientes")
        continue
    a = np.stack(seq)                                  # (pasos, puntos N320)
    mono = float(np.mean(a[1:] >= a[:-1] - 1e-9))     # fracción de puntos no decrecientes
    ratio = float(a[-1].mean() / a[0].mean())
    if mono > 0.99 and ratio > 4:
        verdict = f"acumulado desde t0 -> diferenciar para obtener {v}_6h"
    elif ratio < 2:
        verdict = "por paso de 6 h -> no diferenciar"
    else:
        verdict = "ambiguo: inspeccionar campos"
    print(f"{v}: no decreciente={mono:.3f}, media(último)/media(primero)={ratio:.2f} -> {verdict}")
    print(f"   máximo por paso (mm): {np.round(a.max(axis=1) * 1e3, 1)}")

if not GPU_TRACE.empty:
    print(f"nvidia-smi: memoria máx. {GPU_TRACE.mem_used_gib.max():.1f} GiB, "
          f"utilización media {GPU_TRACE.util_pct.mean():.0f} % ({len(GPU_TRACE)} muestras)")

## 6. Extrapolación a la campaña

Por pronóstico, en estado estacionario:

t_fc = t_entrada + t_t0 + Σ(t_infer + t_post) + t_escritura(esquema)

Por tarea año-mes:

T_tarea = T_fijo + n · t_fc

donde T_fijo es la carga del modelo más el exceso del ciclo 0 sobre la mediana de los ciclos 1–3.

El margen se aplica al total de horas-GPU; `--time` se fija con el máximo de T_tarea × 1.5.

In [ ]:
REC, STP, FIL, CL = (pd.DataFrame(x) for x in (RECORDS, STEPS, FILES, CLTEST))
MARGIN = 1.3

core = REC[REC.phase.isin(["input_state", "post_t0", "forecast"])].groupby("cycle").wall_s.sum()
write = FIL.groupby(["cycle", "layout"]).write_s.sum().unstack("layout")
t_load = REC.loc[REC.phase == "load_checkpoint", "wall_s"].sum()
t_core = core.iloc[1:].median()
t_fix = t_load + core.iloc[0] - t_core
cv = core.iloc[1:].std() / core.iloc[1:].mean()

inits_all = init_dates(range(SCHED["years"][0], SCHED["years"][1] + 1),
                       SCHED["months"][0], SCHED["months"][1], SCHED["init_frequency_hours"])
per_task = np.array([len(v) for v in group_by_year_month(inits_all).values()])
n_an = len(analysis_times(inits_all, LAG_H))

print(f"carga: {t_load:.1f} s | ciclo 0: {core.iloc[0]:.1f} s | estacionario: {t_core:.1f} s (CV {cv:.1%}) | T_fijo: {t_fix:.1f} s")
print(STP[STP.cycle > 0][["infer_s", "post_s"]].describe().loc[["mean", "50%", "max"]].round(2))

SUMMARY = {}
for lay in LAYOUTS:
    t_fc = t_core + write[lay].iloc[1:].median()
    t_task = t_fix + per_task * t_fc
    b_fc = FIL[FIL.layout == lay].groupby("cycle").bytes.sum().mean()
    SUMMARY[lay] = {"s_por_pronostico": t_fc, "tarea_h_max": t_task.max() / 3600,
                    "gpu_h_total": t_task.sum() / 3600, "gpu_h_con_margen": MARGIN * t_task.sum() / 3600,
                    "MB_por_pronostico": b_fc / 1e6, "salida_TB": b_fc * len(inits_all) / 1e12,
                    "compresion": FIL[FIL.layout == lay].bytes.sum() / FIL[FIL.layout == lay].raw_bytes.sum()}
display(pd.DataFrame(SUMMARY).T.round(3))
display(CL.assign(MB=CL.bytes / 1e6).round(3))

# Proyección de tamaño del esquema single para otros límites orientales del dominio extendido
ext = LAYOUTS["single"]["extended"]
bpv = FIL[FIL.layout == "single"].bytes.mean() / (N_T * len(ext["variables"]) * n_points(ext))
n_lat = int(round((max(ext["lat"]) - min(ext["lat"])) / RES)) + 1
for lon_e in (310.0, 320.0, 330.0):
    npts = n_lat * (int(round((lon_e - min(ext["lon"])) / RES)) + 1)
    print(f"single, {min(ext['lon']):.0f}-{lon_e:.0f}E: ~{bpv * N_T * len(ext['variables']) * npts * len(inits_all) / 1e12:.2f} TB")

rss_peak = REC.rss_peak_gib.max()
vram_peak = REC.vram_peak_reserved_gib.max()
smi_peak = GPU_TRACE.mem_used_gib.max() if not GPU_TRACE.empty else np.nan
heavy = REC[REC.phase.isin(["input_state", "post_t0", "write"])]
cores_eff = float((heavy.cpu_s / heavy.wall_s).max())
mem_req = math.ceil(1.3 * rss_peak + 2)
time_req_min = math.ceil(1.5 * max(v["tarea_h_max"] for v in SUMMARY.values()) * 60 / 15) * 15
time_req = f"{time_req_min // 60:02d}:{time_req_min % 60:02d}:00"

print(f"RAM pico (VmHWM): {rss_peak:.1f} GiB -> --mem={mem_req}G")
print(f"VRAM pico reservada (PyTorch): {vram_peak:.1f} GiB | nvidia-smi: {smi_peak:.1f} GiB | total: {ENV['vram_total_gib']}")
print(f"núcleos efectivos en lectura/regrid/escritura: {cores_eff:.1f} (cpus asignadas: {ENV['cpus_affinity']})")
print(f"ERA5: {ERA5_BYTES_PER_TIME * n_an / 1e12:.2f} TB para {n_an} tiempos de análisis; "
      f"~{ERA5_BYTES_PER_TIME * per_task.max() / 1e9:.0f} GB por mes en disco a la vez")
print(f"\n#SBATCH --partition={ENV['slurm']['SLURM_JOB_PARTITION']} --gres=gpu:1 "
      f"--cpus-per-task={max(2, math.ceil(cores_eff) + 1)} --mem={mem_req}G --time={time_req} "
      f"--array=0-{per_task.size - 1}")

In [ ]:
RESULT = {"env": ENV, "summary": SUMMARY, "t_load_s": t_load, "t_fix_s": t_fix, "t_core_s": t_core, "cv_core": cv,
          "rss_peak_gib": rss_peak, "vram_peak_reserved_gib": vram_peak, "smi_peak_gib": smi_peak,
          "era5_bytes_per_time": ERA5_BYTES_PER_TIME, "n_analysis_times": n_an,
          "diag": {"t0_yielded": DIAG["t0_yielded"], "missing_outputs": sorted(DIAG["missing_outputs"]),
                   "input_check": DIAG["input_check"], "regrid_check": DIAG["regrid_check"]}}
(OUT / "summary.json").write_text(json.dumps(RESULT, indent=2, default=str))
for name, df in {"phases": REC, "steps": STP, "files": FIL, "complevel": CL, "gpu_trace": GPU_TRACE}.items():
    df.to_csv(OUT / f"{name}.csv", index=False)
print("resultados en", OUT)